# 🧲 Semantic Routing

**Semantic routing** picks a destination *without asking an LLM*. Each route is described by a few
example questions; an incoming query is embedded and sent wherever it is **closest** in vector
space.

That single design choice defines everything about it: it is roughly 100× cheaper and far faster
than LLM routing — and it cannot reason. It measures resemblance, nothing more.

## Learning Objectives
1. **Routing by distance** — embed example queries once, then route by cosine similarity
2. **Reading the scores** — why the *margin* between routes matters more than the winner
3. **Near-ties** — see a query decided by a 0.002 gap, effectively a coin flip
4. **Where it breaks** — queries phrased unlike any example, and the surface-similarity trap
5. **Choosing between routers** — when distance is enough and when reasoning is required

## Prerequisites
- A `.env` at the repo root with `OPENAI_API_KEY` and `EXPERIENTIALLABS_API_KEY`
- Understanding of embeddings and cosine similarity
- Sibling notebooks: `a. Routing_LLM_Classifier` and `c. Self_Querying_Retrieval`

---
## 🧠 Part 1: Routing Without an LLM

The LLM classifier in `a. Routing_LLM_Classifier` spends a generation call on every query just to
pick a destination. Semantic routing replaces that with arithmetic:

1. **Once, at setup**: embed a handful of example questions for each route.
2. **Per query**: embed the query, compute cosine similarity against every example, take the max.

The only per-query cost is a single embedding call — cheaper and faster than a generation call by
orders of magnitude.

### Key Concepts:
- **Route exemplars**: the example questions that define what a route "looks like".
- **Cosine similarity**: the angle between two embedding vectors; `1.0` is identical, `0.0` unrelated.
- **Margin**: the gap between the winning score and the runner-up. A tiny margin means the
  decision was nearly arbitrary.

> **Key Insight**: semantic routing has no concept of *meaning* beyond surface resemblance to your
> exemplars. It cannot infer that a question about gym membership costs is really about budgeting.
> The quality of your exemplars **is** the quality of your router.

---
## ⚙️ Part 2: Environment Setup

### 2.1 Imports

> **⚠️ Import moved**: the original notebook used `from langchain.utils.math import
> cosine_similarity`, which no longer exists — `langchain.utils` was removed in LangChain 1.x.
> The helper now lives in `langchain_classic.utils.math`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
import os
import warnings

import numpy as np
from dotenv import load_dotenv

# LangChain core
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Integrations
from langchain_openai import OpenAIEmbeddings

# Moved in LangChain 1.x: was langchain.utils.math
from langchain_classic.utils.math import cosine_similarity

# Project helper (LLM factory)
from helpers import get_experientiallabs_llm

warnings.filterwarnings("ignore")

print("✅ Imports loaded successfully!")

### 2.2 Credentials and LangSmith Tracing

> **Note**: use `LANGSMITH_PROJECT`, not the legacy `LANGCHAIN_PROJECT` — the SDK checks the
> `LANGSMITH_` prefix first, so a value in `.env` would otherwise silently win.

In [ ]:
# ============================================================================
# CONFIGURATION: Credentials and tracing
# ============================================================================
load_dotenv()

os.environ["LANGSMITH_PROJECT"] = "Semantic-Routing"

print(f"✅ OpenAI key present:  {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"✅ LangSmith tracing:   {os.getenv('LANGSMITH_TRACING')}")
print(f"✅ LangSmith project:   {os.environ['LANGSMITH_PROJECT']}")

### 2.3 Initialize the Models

Note the asymmetry with LLM routing: here the **embedding model does the routing** and the LLM only
answers. The router never invokes the LLM at all.

In [ ]:
# ============================================================================
# MODEL INITIALIZATION: Embeddings route, the LLM only answers
# ============================================================================
# Pinned explicitly — a bare OpenAIEmbeddings() still defaults to legacy ada-002.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = get_experientiallabs_llm()

print(f"🔢 Embeddings: {embeddings.model}  (does the routing)")
print(f"🤖 LLM:        {llm.model_name}  (only answers)")

---
## 📝 Part 3: Destinations and Their Exemplars

Two things per route: the **specialist prompt** that answers, and the **example questions** that
define what belongs there. Only the exemplars participate in routing — the prompt text is never
embedded.

In [ ]:
# ============================================================================
# DESTINATIONS: Specialist prompts
# ============================================================================
personal_finance_template = """You are a personal finance expert with extensive knowledge of budgeting, investing, and financial planning. You offer clear and practical advice on managing money and making sound financial decisions.

Here is a question:
{query}"""

book_review_template = """You are an experienced book critic with extensive knowledge of literature, genres, and authors. You provide thoughtful and analytical reviews and insights about books.

Here is a question:
{query}"""

health_fitness_template = """You are a certified health and fitness expert with a deep understanding of nutrition, exercise routines, and wellness strategies. You offer practical and evidence-based advice about health and fitness.

Here is a question:
{query}"""

travel_guide_template = """You are a seasoned travel expert with extensive knowledge of destinations, travel tips, and cultural insights. You provide detailed and useful advice about travel.

Here is a question:
{query}"""

print("✅ 4 specialist prompts defined")

### 3.1 Route Exemplars

These example questions *are* the router's definition of each domain. Three per route is thin for
production — more exemplars, covering more phrasings, mean better coverage.

In [ ]:
# ============================================================================
# EXEMPLARS: The example questions that define each route
# ============================================================================
ROUTES = {
    "personal_finance": {
        "template": personal_finance_template,
        "examples": [
            "What are the best strategies for saving money?",
            "How do I start investing in the stock market?",
            "What should I consider when creating a budget?",
        ],
    },
    "book_review": {
        "template": book_review_template,
        "examples": [
            "What makes a novel a classic?",
            "How do you analyze the themes in a book?",
            "What are the key differences between literary fiction and genre fiction?",
        ],
    },
    "health_fitness": {
        "template": health_fitness_template,
        "examples": [
            "What are the benefits of a balanced diet?",
            "How often should I exercise to maintain good health?",
            "What are effective strategies for losing weight?",
        ],
    },
    "travel_guide": {
        "template": travel_guide_template,
        "examples": [
            "What are the must-see attractions in Tokyo?",
            "How can I travel on a budget?",
            "What should I know before traveling to a foreign country?",
        ],
    },
}

total = sum(len(r["examples"]) for r in ROUTES.values())
print(f"✅ {len(ROUTES)} routes, {total} exemplar questions total")

---
## 🔢 Part 4: Embed the Exemplars — Once

This is the entire setup cost, paid a single time. In production you would cache these vectors
rather than recompute them at every start-up.

In [ ]:
# ============================================================================
# INDEX: Embed every exemplar once, up front
# ============================================================================
for name, route in ROUTES.items():
    route["embeddings"] = embeddings.embed_documents(route["examples"])

dims = len(ROUTES["personal_finance"]["embeddings"][0])
print(f"✅ Embedded {total} exemplars into {dims}-dimensional vectors")
print("   Per-query cost from here on: ONE embedding call, no LLM.")

---
## 🧮 Part 5: Scoring a Query

The routing rule is one line of arithmetic: **each route scores as the maximum similarity between
the query and any of its exemplars**, and the highest score wins.

The scoring function below returns *all* scores rather than just the winner. That matters — a
router that only tells you its answer is impossible to debug.

> **Note**: the original implementation compared floats with `max_similarity == max(...)` to pick
> the winner. That works, but it recomputes the maxima and silently ties-breaks by branch order.
> Sorting the scores makes both the winner and the **margin** explicit.

In [ ]:
# ============================================================================
# SCORING: Similarity of a query against every route
# ============================================================================
def score_routes(query: str) -> list[tuple[str, float]]:
    """Return (route_name, best_similarity) for every route, highest first."""
    query_embedding = embeddings.embed_query(query)
    scores = [
        (name, float(max(cosine_similarity([query_embedding], route["embeddings"])[0])))
        for name, route in ROUTES.items()
    ]
    return sorted(scores, key=lambda pair: pair[1], reverse=True)


def explain_routing(query: str) -> str:
    """Print the full score table for a query and return the winning route."""
    scores = sorted(score_routes(query), key=lambda p: p[1], reverse=True)
    winner, top = scores[0]
    margin = top - scores[1][1]

    print(f"❓ {query}")
    for rank, (name, score) in enumerate(scores):
        bar = "█" * int(score * 40)
        marker = "🏆" if rank == 0 else "  "
        print(f"   {marker} {name:<18} {score:.4f} {bar}")
    print(f"   📏 margin over runner-up: {margin:.4f}", end="")
    print("   ⚠️  NEAR TIE — essentially arbitrary\n" if margin < 0.02 else "\n")
    return winner


print("✅ Scoring helpers ready")

---
## 🔬 Part 6: Look Inside the Router

Three cases, in increasing order of instructiveness.

### 6.1 An Exact Match

*"How can I travel on a budget?"* is verbatim one of the travel exemplars, so it scores `1.0000`.
Note that `personal_finance` still scores ~0.50 — the words *budget* and *money* pull it there too.

In [ ]:
# ============================================================================
# CASE 1: Query identical to an exemplar
# ============================================================================
explain_routing("How can I travel on a budget?")

### 6.2 A Clear but Non-Identical Match

Rephrased entirely, yet still comfortably travel. This is semantic routing working as intended —
matching *meaning*, not keywords.

In [ ]:
# ============================================================================
# CASE 2: Novel phrasing, still a confident decision
# ============================================================================
explain_routing("What is the cheapest way to see Europe without overspending?")

### 6.3 The Instructive Case — A Near Tie

*"Are meal-prep books worth buying?"* is the one to study.

Three things go wrong at once, and every one of them is characteristic of semantic routing:

1. **The margin is ~0.002** — the winner beats the runner-up by a rounding error. Re-run it with
   slightly different exemplars and the decision flips.
2. **Every score is low (~0.28)** — nothing matched well. The router has no way to say *"none of
   these"*; it returns the least-bad option with the same confidence it uses for a perfect match.
3. **`book_review` is not even in the top two**, despite the query literally containing *"books"* —
   the query's *overall* meaning leans toward purchasing decisions and meal prep.

An LLM classifier would likely reason its way to a defensible answer here. Distance cannot.

In [ ]:
# ============================================================================
# CASE 3: Near tie — the decision is effectively a coin flip
# ============================================================================
explain_routing("Are meal-prep books worth buying?")

### 6.4 Quantifying Confidence

A practical guard: treat a small margin, or a low top score, as *"unsure"* and fall back to an LLM
classifier or a default prompt. The threshold below is illustrative — tune it on your own traffic.

In [ ]:
# ============================================================================
# CONFIDENCE GUARD: Detect unreliable routing decisions
# ============================================================================
MIN_SCORE = 0.35   # nothing matched well enough
MIN_MARGIN = 0.02  # winner and runner-up too close to distinguish

for q in [
    "How do I start investing in the stock market?",
    "Are meal-prep books worth buying?",
    "How do I save money while travelling abroad?",
]:
    scores = score_routes(q)
    (winner, top), (_, second) = scores[0], scores[1]
    margin = top - second

    if top < MIN_SCORE:
        verdict = f"⚠️  LOW CONFIDENCE (top score {top:.4f} < {MIN_SCORE})"
    elif margin < MIN_MARGIN:
        verdict = f"⚠️  AMBIGUOUS (margin {margin:.4f} < {MIN_MARGIN})"
    else:
        verdict = f"✅ CONFIDENT -> {winner}"

    print(f"{verdict}\n   {q}\n   top={top:.4f}  margin={margin:.4f}\n")

---
## 🔗 Part 7: The Full Routed Chain

`RunnableLambda` places the router inside an LCEL chain. It returns the **selected prompt**, which
the LLM then executes — the same shape as the LLM-classifier notebook, but with a router that never
calls a model.

In [ ]:
# ============================================================================
# ROUTED CHAIN: score -> select prompt -> answer
# ============================================================================
def prompt_router(input_dict):
    """Pick the closest route's prompt for this query."""
    winner, score = score_routes(input_dict["query"])[0]
    print(f"🧲 Routed to: {winner}  (similarity {score:.4f})")
    return PromptTemplate.from_template(ROUTES[winner]["template"])


full_chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

print("✅ Routed chain ready")

### 7.1 One Query Per Destination

The original notebook repeated four near-identical cells for this; a loop makes the comparison
easier to read.

In [ ]:
# ============================================================================
# END TO END: One query per destination
# ============================================================================
queries = [
    "What are the must-see attractions in the USA?",
    "What makes a novel a classic?",
    "What are effective strategies for losing weight?",
    "What are the best strategies for saving money?",
]

for q in queries:
    print(f"\n{'=' * 78}\n❓ {q}\n{'=' * 78}")
    print(full_chain.invoke(q)[:500], "...")

---
## 📝 Summary

### 1. How Semantic Routing Works
- Embed a few **exemplar questions** per route once; per query, embed and take the nearest.
- Per-query cost is **one embedding call** — no LLM in the routing path at all.

### 2. Read the Margin, Not Just the Winner
- The winning score alone is not confidence. *"Are meal-prep books worth buying?"* won by a
  **0.002 margin** with every score around 0.28 — a decision indistinguishable from a coin flip.
- Guard with two thresholds: a **minimum top score** (did anything match?) and a **minimum margin**
  (was the choice distinguishable?). Fall back to an LLM classifier when either fails.

### 3. The Exemplars Are the Router
- Quality and coverage of example questions fully determine routing quality. Thin exemplars produce
  low scores across the board and arbitrary winners.

### 4. What It Cannot Do
- It measures **resemblance**, not meaning. A query containing *"books"* routed to
  `personal_finance` because its overall phrasing leaned toward purchasing decisions.
- There is no *"none of these"*. Out-of-scope queries are routed with the same machinery as
  perfect matches.

### 5. Environment Note
- `from langchain.utils.math import cosine_similarity` no longer exists in LangChain 1.x. It moved
  to `langchain_classic.utils.math`.

### 6. How This Compares to Its Siblings
| Notebook | Routes on | Decides by | Cost per query |
|---|---|---|---|
| **`a. Routing_LLM_Classifier`** | Which **prompt/data source** | LLM reasoning | One extra LLM call |
| **`b. Semantic_Routing`** (this one) | Which **prompt** | Embedding distance to exemplars | One embedding call (~100× cheaper) |
| **`c. Self_Querying_Retrieval`** | Nothing — one source | Builds a **metadata filter** instead | One LLM call |

- **Use semantic routing** when destinations are well separated, traffic is high, and latency or
  cost matter. It is deterministic and trivially debuggable — the scores explain every decision.
- **Use LLM routing** when queries are nuanced, multi-clause, or phrased unpredictably, and when
  you need an *"other"* escape hatch. It reasons; it also costs more and can hallucinate a route.
- **A strong production pattern is both**: route semantically, and escalate to the LLM classifier
  only when the margin is too small — which is exactly what Part 6.4 detects.

### Next Steps
- Inspect these runs in LangSmith under the **Semantic-Routing** project. Notice the routing step
  produces no LLM call — only the answer does.
- Run the same ambiguous queries through `a. Routing_LLM_Classifier` and compare the decisions.